<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2006%20-%20Derivatives%3A%20The%20Compass%20for%20Learning/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 06 — Derivatives: The Compass for Learning · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

A car's distance after $t$ seconds is $s(t) = t^2$ metres. The speedometer shows a number at
every instant — but at a single instant the car covers **0 metres in 0 seconds**.

$$\text{speed at an instant} = \frac{0}{0}\;?$$

Chapter 1 did exactly this to find its compass: it divided by a gap and then made the gap
vanish. This lab checks whether that was legitimate.

## Step 2 — Prediction

Commit before running. You check these in Step 10.

1. Average speed of $s(t)=t^2$ from $t=2$ to $t=2+h$, for $h = 1, 0.5, 0.1, 0.01$. What
   number are these heading towards?
2. We know $\frac{d}{dx}x^2 = 2x$, so at $x=2$ the true slope is exactly 4. If a computer
   estimates it with smaller and smaller $h$, does the error keep shrinking all the way
   down to $h = 10^{-14}$?
3. Chapter 1 found training exploded near $\eta \approx 0.13$. The loss is
   $L(w) = 7.5(w-2)^2$. Can you predict the exact threshold from its curvature?
4. What is the slope of $|x|$ at $x = 0$?

In [ ]:
# Step 3 — Intuition: shrink the gap and watch where the answer heads.
import numpy as np
np.random.seed(0)

def s(t):
    return t ** 2                      # distance in metres after t seconds

print("average speed from t=2 to t=2+h")
print(f"{'h':>10} {'avg speed':>12} {'= 4 + h?':>12}")
for h in [1.0, 0.5, 0.1, 0.01, 0.001]:
    avg = (s(2 + h) - s(2)) / h
    print(f"{h:>10} {avg:>12.6f} {4 + h:>12.6f}")
    assert np.isclose(avg, 4 + h)      # the algebra in blog section 4

# Never once did we divide by zero. We only watched a trend.
print("\nThe answers march towards 4. That is the speed at t = 2.")

## Step 4 — The Mathematics Under Test

$$f'(x) = \lim_{h \to 0}\frac{f(x+h)-f(x)}{h}
\qquad
\frac{d}{dx}x^n = n x^{n-1}
\qquad
\eta < \frac{2}{f''}$$

Step 5 checks every number the lecture claims.

In [ ]:
# Step 5 — Manual calculation: the rules derived in blog section 7.
def numeric_derivative(f, x, h=1e-6):
    """The definition, with a small but not tiny h (see Step 8 for why 1e-6)."""
    return (f(x + h) - f(x)) / h

# d/dx x^2 = 2x
for x in [2.0, 3.0, -1.5]:
    assert np.isclose(numeric_derivative(lambda v: v**2, x), 2 * x, atol=1e-5)
print("d/dx x^2 = 2x   confirmed at x = 2, 3, -1.5")

# d/dx x^3 = 3x^2
for x in [2.0, 3.0]:
    assert np.isclose(numeric_derivative(lambda v: v**3, x), 3 * x**2, atol=1e-4)
print("d/dx x^3 = 3x^2 confirmed")

# the power rule in general
for n in [2, 3, 4, 5]:
    x = 1.7
    assert np.isclose(numeric_derivative(lambda v: v**n, x), n * x**(n - 1), atol=1e-4)
print("power rule d/dx x^n = n x^(n-1) confirmed for n = 2..5")

# a constant has zero slope; constants factor out; sums add
assert np.isclose(numeric_derivative(lambda v: 7.0, 3.0), 0.0, atol=1e-9)
assert np.isclose(numeric_derivative(lambda v: 5 * v**2, 2.0), 5 * 2 * 2.0, atol=1e-4)
assert np.isclose(numeric_derivative(lambda v: v**2 + v**3, 2.0),
                  2 * 2.0 + 3 * 2.0**2, atol=1e-4)
print("constant, constant-multiple and sum rules confirmed")

In [ ]:
# Step 5b — pay Chapter 1's debt: derive its compass properly (blog section 8).
def L(w):
    return 7.5 * (w - 2) ** 2          # Chapter 1's loss, bias pinned at 1

def L_prime(w):
    return 15 * (w - 2)                # what we derived with the rules

print(f"{'w':>5} {'L(w)':>10} {'L\\'(w)':>10} {'numeric':>12}   meaning")
for w in [0.0, 1.0, 2.0, 3.0, 4.0]:
    num = numeric_derivative(L, w)
    assert np.isclose(num, L_prime(w), atol=1e-4)
    if L_prime(w) < 0:
        meaning = "downhill to the right -> increase w"
    elif L_prime(w) > 0:
        meaning = "uphill to the right   -> decrease w"
    else:
        meaning = "flat - the bottom"
    print(f"{w:>5} {L(w):>10.2f} {L_prime(w):>10.1f} {num:>12.5f}   {meaning}")

# Chapter 1's own hand-computed loss table must still match.
assert np.isclose(L(0), 30.0) and np.isclose(L(1), 7.5) and np.isclose(L(2), 0.0)
print("\nChapter 1's table reproduced, and its compass 15(w-2) is now derived, not asserted.")

In [ ]:
# Step 6 — First implementation: the minus sign in the update rule needs no if-statement.
def descend(start, eta, steps=200):
    w = start
    for _ in range(steps):
        w = w - eta * L_prime(w)       # ONE rule handles both directions
        if not np.isfinite(w) or abs(w) > 1e12:
            return float("nan")
    return w

print("starting left of the minimum: ", round(descend(0.0, 0.1), 6))
print("starting right of the minimum:", round(descend(5.0, 0.1), 6))
assert np.isclose(descend(0.0, 0.1), 2.0, atol=1e-6)
assert np.isclose(descend(5.0, 0.1), 2.0, atol=1e-6)

# From the left the slope is negative, so subtracting it INCREASES w.
# From the right it is positive, so subtracting it DECREASES w. Same line of code.

In [ ]:
# Step 7 — Visualization: a secant becoming a tangent (blog section 9).
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) secants closing in on the tangent at t = 2
t = np.linspace(0, 4, 200)
axes[0].plot(t, s(t), lw=2, label="s(t) = t²")
for h, col in zip([1.5, 1.0, 0.5], ["#cce", "#88b", "#448"]):
    slope = (s(2 + h) - s(2)) / h
    axes[0].plot(t, s(2) + slope * (t - 2), color=col, lw=1,
                 label=f"secant h={h} (slope {slope:.2f})")
axes[0].plot(t, s(2) + 4 * (t - 2), color="crimson", lw=2, label="tangent (slope 4)")
axes[0].plot(2, s(2), "o", color="crimson", ms=8)
axes[0].set_ylim(-2, 16); axes[0].set_title("secant → tangent")
axes[0].legend(fontsize=7); axes[0].grid(alpha=.3)

# (b) the loss and its derivative side by side
w = np.linspace(-0.5, 4.5, 200)
axes[1].plot(w, L(w), label="L(w)")
axes[1].plot(w, L_prime(w), label="L'(w) = 15(w-2)")
axes[1].axhline(0, color="grey", lw=.8)
axes[1].axvline(2, color="crimson", ls="--", lw=1)
axes[1].set_title("where L' crosses zero, L bottoms out")
axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)

# (c) |x| has no derivative at 0
x = np.linspace(-2, 2, 400)
axes[2].plot(x, np.abs(x), lw=2)
axes[2].plot([-2, 0], [-(-2), 0], "--", color="crimson", lw=1, label="slope -1 on the left")
axes[2].plot([0, 2], [0, 2], "--", color="darkgreen", lw=1, label="slope +1 on the right")
axes[2].plot(0, 0, "o", color="black", ms=7)
axes[2].set_title("|x|: the two sides never agree")
axes[2].legend(fontsize=8); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

In [ ]:
# Step 8 — The experiment: is a smaller h always better? (blog section 11)
true_slope = 4.0                      # d/dx x^2 at x = 2

print(f"{'h':>10} {'approximation':>20} {'error':>14}")
results = []
for power in range(1, 16):
    h = 10.0 ** (-power)
    approx = ((2 + h) ** 2 - 2 ** 2) / h
    err = abs(approx - true_slope)
    results.append((h, err))
    print(f"{h:>10.0e} {approx:>20.12f} {err:>14.2e}")

best_h, best_err = min(results, key=lambda r: r[1])
print(f"\nbest h = {best_h:.0e}  with error {best_err:.2e}")

# The error does NOT keep shrinking: it bottoms out and then gets worse.
assert 1e-9 < best_h < 1e-6            # the sweet spot, near sqrt(machine epsilon)
assert results[-1][1] > best_err * 1000  # h=1e-15 is far worse than the best
print(f"machine epsilon = {np.finfo(float).eps:.2e},  sqrt = {np.sqrt(np.finfo(float).eps):.2e}")

In [ ]:
# Step 9 — Change exactly one variable: the learning rate, against the curvature limit.
# L''(w) = 15 everywhere, so the theory says eta < 2/15.
limit = 2 / 15
print(f"L'' = 15  ->  predicted stability limit eta < 2/15 = {limit:.6f}\n")

print(f"{'eta':>10} {'w after 200 steps':>22}   verdict")
for eta in [0.10, 0.13, 0.1333, 0.134, 0.15]:
    with np.errstate(over="ignore", invalid="ignore"):
        w_end = descend(0.0, eta)
    if not np.isfinite(w_end):
        print(f"{eta:>10} {'overflow':>22}   diverged")
    else:
        ok = "converged" if abs(w_end - 2) < 1e-3 else "unstable"
        print(f"{eta:>10} {w_end:>22.6f}   {ok}")

assert np.isclose(descend(0.0, 0.13), 2.0, atol=1e-3)     # below the limit: fine
assert not np.isclose(descend(0.0, 0.15), 2.0, atol=1e-3) # above the limit: not fine

## Step 10 — Observe

Against your Step 2 predictions:

1. The average speeds are exactly $4 + h$: 5, 4.5, 4.1, 4.01 — marching to **4**. Not
   approximately; the algebra in §4 says the average over *any* gap $h$ is $4+h$.
2. **No.** The error shrinks down to about $h = 10^{-8}$ and then *grows again*. At
   $h = 10^{-14}$ it is as bad as it was at $h = 10^{-1}$.
3. The curvature is $L'' = 15$, so the limit is $2/15 = 0.1333$. And the runs bracket it:
   $\eta = 0.13$ converges, $\eta = 0.134$ drifts away, $\eta = 0.15$ overflows.
4. $|x|$ has **no** slope at 0 — the right side says $+1$, the left says $-1$, forever.

## Step 11 — Explain

**Why the limit is legal.** At every step $h$ was a real non-zero number, so every division
was genuine. We never asked what happens *at* $h=0$; we asked where the answers were
**heading**. Cancelling $h$ before letting it shrink is what makes the difference between
$0/0$ and $4$.

**Why the numerical error turns around.** Two errors fight:

$$\text{error} \approx \underbrace{h}_{\text{mathematics}} + \underbrace{\varepsilon/h}_{\text{machine}}$$

Truncation shrinks with $h$; rounding grows as $h$ shrinks, because subtracting two nearly
equal numbers destroys significant digits and then we divide the wreckage by something tiny.
They balance near $h \approx \sqrt{\varepsilon} \approx 10^{-8}$, which is exactly where
Step 8 bottoms out.

> Calculus says smaller $h$ is always better. Floating point says not below $10^{-8}$.
> Both are right about their own world, and you have to know both.

**Why curvature limits the learning rate.** A gradient step assumes the slope stays roughly
constant over the distance travelled. Curvature measures how wrong that assumption gets. With
$L''=15$, a step longer than $2/15$ lands where the slope has changed so much that you
overshoot — and the overshoot compounds. Chapter 1 measured that threshold experimentally;
here we predicted it from the second derivative before running anything.

In [ ]:
# Step 12 — Challenges.

# LEVEL 2 (by hand first): f(x) = x^2 - 6x + 5.
# Find f'(x), the x where f'(x)=0, and use f'' to classify it. Then check:
f = lambda x: x**2 - 6*x + 5
print("f'(3) =", round(numeric_derivative(f, 3.0), 6), " (should be 0)")
assert abs(numeric_derivative(f, 3.0)) < 1e-4

# LEVEL 4 (Investigate): redo the Step 8 error sweep at x = 1000 instead of x = 2.
# The best h moves. Explain why, using the fact that rounding error depends on the
# SIZE of the numbers being subtracted, not on h alone.

# YOUR CODE HERE


# LEVEL 5 (Design): the forward difference has error about h.
# Build a better estimator from f(x+h) and f(x-h), work out its error by hand first,
# then measure it here. Why is it more accurate, and what does it cost?
def central_difference(f, x, h=1e-6):
    # YOUR CODE HERE
    ...

# Once written, compare its best achievable error against Step 8's.

## Step 13 — Reflection

- [ ] I can explain why $0/0$ is not the answer, and what question we asked instead.
- [ ] I derived $\frac{d}{dx}x^2 = 2x$ from the definition, cancelling $h$ before shrinking it.
- [ ] I can rebuild the power rule without looking it up.
- [ ] I can say why the update rule has a minus sign and needs no `if`.
- [ ] I know why $h = 10^{-14}$ is worse than $h = 10^{-8}$.
- [ ] I can predict a learning-rate limit from a second derivative.
- [ ] I can say what ReLU and $|x|$ have in common at the origin.

### The question this chapter leaves open

Everything here differentiated a function of **one** variable. To get $L(w) = 7.5(w-2)^2$,
Chapter 1 had to pin the bias at $b = 1$.

The real loss depends on $w$ **and** $b$. That surface is a bowl, not a curve — walk east and
it rises, walk north and it falls. "How steep is it here?" is not even a complete question
until you say *in which direction*. And a real network has a billion dials, not two.

➡️ **Next:** [Chapter 07 — Partial Derivatives, Gradients and the Chain Rule](<../Lecture 07 - Partial Derivatives, Gradients and the Chain Rule/blog.md>)